# 1. Pipeline Initialization & Artifact Ingestion

This cell sets up the environment to run the machine learning model by loading our saved predictive pipeline and preparing the testing data.

* **Reconstruct:** Pulls in and merges patient records using the same pipeline script used during model training.
* **Load Artifacts:** Ingests the saved stacking ensemble model and the customized 0.35 risk threshold.
* **Predict:** Calculates raw readmission probabilities (`stacking_probs`) and applies the threshold to generate final predictions.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
import warnings
warnings.filterwarnings("ignore")

base_dir = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
reports_dir = os.path.join(base_dir, "reports")
os.makedirs(reports_dir, exist_ok=True)

# CSV paths
diagnoses_path = os.path.join(base_dir, "data", "diagnoses.csv")
patients_path  = os.path.join(base_dir, "data", "patients.csv")
outcomes_path  = os.path.join(base_dir, "data", "outcomes.csv")

from src.data_pipeline import load_and_merge_data, clean_and_impute
from src.features import engineer_features

# Loading Data
df = load_and_merge_data(diagnoses_path, patients_path, outcomes_path)
admitted = clean_and_impute(df)
X, y, feature_cols = engineer_features(admitted)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Loading Artifacts
stacking_model = joblib.load("../models/best_model.pkl")
best_threshold = joblib.load("../models/best_threshold.pkl")

# Extracting base XGBoost model from the Stacking Classifier for exact Tree SHAP
xgb_model = stacking_model.named_estimators_["xgb"]

stacking_probs = stacking_model.predict_proba(X_test)[:, 1]
stacking_preds = (stacking_probs >= best_threshold).astype(int)

# 2. SHAP Explainer Initialization

This cell sets up our Explainable AI (XAI) framework to calculate how individual features affect the model's predictions.

* **Explainer:** Initializes a SHAP tree engine tailored specifically for tree-based models like XGBoost.
* **Calculate:** Computes Shapley values (`shap_values`) across our entire holdout testing set.
* **Verify:** Prints the final data shapes to confirm the impact scores align perfectly with our patient count.

In [ ]:
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_test)
sv           = shap_values.values

print(f"SHAP values shape: {sv.shape}")
print(f"Features: {X_test.shape[1]}  |  Test samples: {X_test.shape[0]}")

# 3. Global Feature Importance Matrix

This cell ranks patient metrics and diagnoses to show what drives hospital readmissions across the entire cohort.

* **Calculate:** Extracts the average absolute impact score for every clinical variable in the data.
* **Rank:** Lists and prints the top 20 most influential features on risk scores.
* **Plot:** Generates a horizontal bar chart so you can instantly see which factors dominate the model's decision-making process.

In [ ]:
mean_abs_shap = np.abs(sv).mean(axis=0)
importance_df = pd.DataFrame({
    "feature":    feature_cols,
    "importance": mean_abs_shap,
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("\nTop 20 features by mean |SHAP|:")
print(importance_df.head(20).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 7))
top20 = importance_df.head(20)
bars  = ax.barh(
    top20["feature"][::-1],
    top20["importance"][::-1],
    color="#378ADD", edgecolor="none",
)
ax.set_xlabel("Mean |SHAP value| — average impact on readmission probability")
ax.set_title("Top 20 features driving 30-day readmission (XGBoost SHAP)")
ax.axvline(0, color="gray", lw=0.5)
plt.tight_layout()
plt.show()

# 4. Global Distribution & Directional Impact

This cell evaluates how high or low values of a specific feature push a patient's readmission risk score up or down.

* **Direction:** Uses a beeswarm plot to show if a feature increases or decreases risk (e.g., high comorbidity burden vs. low burden).
* **Magnitude:** Displays the spread and density of risk impacts across thousands of individual patient records.
* **Save:** Automatically exports the primary visual summary chart to our `reports/` folder for documentation.

In [ ]:
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.title("SHAP beeswarm — feature impact direction and magnitude")
plt.tight_layout()
plt.savefig(os.path.join(reports_dir, 'shap_global_summary.png'), bbox_inches='tight', dpi=300)
plt.show()

shap.summary_plot(sv, X_test, feature_names=feature_cols, max_display=20, show=False)
plt.title("SHAP summary — readmission risk drivers")
plt.tight_layout()
plt.show()

# 5. Non-Linear Feature Dependence Curves

This cell drills into our top 4 most important variables to see exactly how changing their values shifts a patient's risk profile.

* **Isolate:** Visualizes single features on a scatter grid to trace individual patient changes.
* **Interact:** Automatically identifies and color-codes a second interacting feature to see how variables mix.
* **Thresholds:** Spots non-linear patterns, highlighting the exact medical tipping points where risk sharply accelerates.

In [ ]:
top4 = importance_df["feature"].head(4).tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, feat in zip(axes.flat, top4):
    feat_idx = feature_cols.index(feat)
    shap.dependence_plot(
        feat_idx, sv, X_test,
        feature_names=feature_cols,
        ax=ax, show=False,
    )
    ax.set_title(f"SHAP dependence — {feat}")

plt.suptitle("How top features affect readmission risk", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# 6. Clinical Case Studies (Individual Patient Diagnostics)

This cell isolates individual patient records at opposing extremes to show how the model builds unique risk scores for different people.

* **Isolate:** Automatically extracts the highest-risk and lowest-risk patients from the test group.
* **Decompose:** Uses waterfall plots to show exactly which medical features push a single patient's score up or down from the baseline.
* **Save:** Exports both personalized clinical audit logs as images to our `reports/` folder.

In [ ]:
high_risk_idx   = np.argmax(stacking_probs)

print(f"\nHighest-risk patient index: {high_risk_idx}")
print(f"  Stacking predicted prob : {stacking_probs[high_risk_idx]:.3f}")
print(f"  Actual label            : {int(y_test.iloc[high_risk_idx])}")

fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.waterfall(shap_values[high_risk_idx], max_display=15, show=False)
plt.title(f"Patient #{high_risk_idx} — high-risk waterfall (predicted prob: {stacking_probs[high_risk_idx]:.2f})")
plt.tight_layout()
plt.savefig(os.path.join(reports_dir, 'high_risk_waterfall.png'), bbox_inches='tight', dpi=300)
plt.show()

low_risk_idx = np.argmin(stacking_probs)

print(f"\nLowest-risk patient index: {low_risk_idx}")
print(f"  Stacking predicted prob : {stacking_probs[low_risk_idx]:.3f}")
print(f"  Actual label            : {int(y_test.iloc[low_risk_idx])}")

fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.waterfall(shap_values[low_risk_idx], max_display=15, show=False)
plt.title(f"Patient #{low_risk_idx} — low-risk waterfall (predicted prob: {stacking_probs[low_risk_idx]:.2f})")
plt.tight_layout()
plt.savefig(os.path.join(reports_dir, 'low_risk_waterfall.png'), bbox_inches='tight', dpi=300)
plt.show()

# 7. System-Wide Performance Validation & Error Analysis

This final cell evaluates our calibrated 0.35 decision threshold across the entire holdout testing group.

* **Metrics:** Calculates and prints standard performance scoring (Accuracy, Precision, Recall, F1) and saves a raw text log.
* **Confusion Matrix:** Plots the exact counts of correct predictions versus system mistakes (False Negatives and False Positives).
* **Calibration:** Generates a reliability curve to verify that our model's risk percentages accurately reflect real-world outcomes.
* **Error Profiling:** Computes and compares average medical profiles for our calculation errors to uncover hidden patterns or edge cases.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# 1. Calculate Core Performance Metrics
accuracy  = accuracy_score(y_test, stacking_preds)
precision = precision_score(y_test, stacking_preds)
recall    = recall_score(y_test, stacking_preds)
f1        = f1_score(y_test, stacking_preds)

metrics_output = (
    "── Core Model Performance Metrics ─────────────────────────────────────\n"
    f"Accuracy  : {accuracy:.4f}\n"
    f"Precision : {precision:.4f}  (Of all patients flagged, how many actually readmitted)\n"
    f"Recall    : {recall:.4f}     (Of all patients who readmitted, how many did we catch)\n"
    f"F1-Score  : {f1:.4f}        (Harmonic mean of precision and recall)\n\n"
    "Detailed Classification Report:\n"
    f"{classification_report(y_test, stacking_preds)}"
)

print(metrics_output)

report_text_path = os.path.join(reports_dir, 'classification_report.txt')
with open(report_text_path, 'w') as f:
    f.write(metrics_output)

# 2. Confusion Matrix
cm = confusion_matrix(y_test, stacking_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Readmit (0)", "Readmit (1)"])

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap="Purples", ax=ax, values_format="d")
plt.title("Readmission Predictor — Confusion Matrix", fontsize=13, pad=15)
plt.grid(False)
plt.tight_layout()
plt.savefig(os.path.join(reports_dir, 'confusion_matrix.png'), bbox_inches='tight', dpi=300)
plt.show()

# 3. Calibration Curve
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(7, 6))
frac_pos, mean_pred = calibration_curve(y_test, stacking_probs, n_bins=10)
ax.plot(mean_pred, frac_pos, marker="o", lw=1.5, color="#7F77DD", label="Stacking (RF+XGB)", markersize=4)

ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4, label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives (actual rate)")
ax.set_title("Calibration curve — are predicted probabilities trustworthy?")
ax.legend(fontsize=9)
plt.tight_layout()

# SAVE THE CALIBRATION CURVE PLOT
plt.savefig(os.path.join(reports_dir, 'calibration_curve.png'), bbox_inches='tight', dpi=300)
plt.show()

# 4. Error Analysis
y_test_arr = y_test.values

false_negatives = X_test[(stacking_preds == 0) & (y_test_arr == 1)]
false_positives = X_test[(stacking_preds == 1) & (y_test_arr == 0)]
true_positives  = X_test[(stacking_preds == 1) & (y_test_arr == 1)]

print("\n── Error analysis ─────────────────────────────────────────────────────")
print(f"False negatives (missed readmissions) : {len(false_negatives)}")
print(f"False positives (false alarms)        : {len(false_positives)}")
print(f"True positives  (correct flags)       : {len(true_positives)}")

compare_cols = ["age", "charlson_index", "length_of_stay_days", "num_comorbidities", "icu_admission"]

summary = pd.DataFrame({
    "False negatives": false_negatives[compare_cols].mean(),
    "False positives": false_positives[compare_cols].mean(),
    "True positives":  true_positives[compare_cols].mean(),
}).round(2)

print("\nAverage profile of each error type:")
print(summary.to_string())